In [4]:
from dotenv import load_dotenv
import os

# Load .env file
load_dotenv()

def _set_environment_variables(
    provider: str = "google",
    api_key: str = None,
    langchain_api_key: str = None,
    langchain_project: str = None,
    langchain_tracing_v2: str = "true",
    langchain_endpoint: str = "https://api.smith.langchain.com",
):
    """
    Loads and sets required environment variables for the application.

    Args:
        provider (str): The API provider, e.g., 'google', 'openai', etc.
        api_key (str, optional): API key for the selected provider. Defaults to value from environment.
        langchain_api_key (str, optional): LangChain API key. Defaults to value from environment.
        langchain_project (str, optional): LangChain project name. Defaults to value from environment or 'code-gen-testing'.
        langchain_tracing_v2 (str, optional): LangChain tracing flag.
        langchain_endpoint (str, optional): LangChain endpoint URL.
    """
    provider_env_map = {
        "google": "GOOGLE_API_KEY",
        "openai": "OPENAI_API_KEY",
        "mistral": "MISTRAL_API_KEY",
        # Add more providers and their env variable names as needed
    }
    env_var = provider_env_map.get(provider.lower())
    if env_var:
        os.environ[env_var] = api_key or os.getenv(env_var, "")
    os.environ["LANGCHAIN_API_KEY"] = langchain_api_key or os.getenv(
        "LANGCHAIN_API_KEY", ""
    )
    os.environ["LANGCHAIN_PROJECT"] = langchain_project or os.getenv(
        "LANGCHAIN_PROJECT", "genai-project"
    )
    os.environ["LANGCHAIN_TRACING_V2"] = langchain_tracing_v2
    os.environ["LANGCHAIN_ENDPOINT"] = langchain_endpoint

In [5]:
_set_environment_variables()

In [3]:
from typing import Any
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mistralai import ChatMistralAI



DEFAULT_MODELS = {
    "openai": ["gpt-3.5-turbo"],
    "google": ["gemini-2.0-flash-001"],
    "mistral": ["mistral-large-latest"],
}

def load_chat_model(model_name: str = None, **kwargs) -> Any:
    """
    Load a chat model using LangChain wrappers.
    Supports OpenAI, Google, and Mistral models.
    If no model_name is specified, uses the default model for each provider.

    Args:
        model_name (str, optional): Model identifier, e.g., "openai/gpt-4.1-mini", "google/gemini-pro", "mistral/mistral-7b".

    Returns:
        Any: An instance of the loaded LangChain chat model.
    """
    if model_name is None:
        raise ValueError(
            "No model_name specified. Please provide a model_name in the format 'provider/model-id', "
            "e.g., 'openai/gpt-3.5-turbo', or use 'openai', 'google', or 'mistral' to use the default."
        )

    if "/" not in model_name:
        provider = model_name.lower()
        if provider in DEFAULT_MODELS:
            model_id = DEFAULT_MODELS[provider][0]
            model_name = f"{provider}/{model_id}"
        else:
            raise ValueError(f"Unknown model provider '{provider}'.")

    if model_name.startswith("openai/"):
        model_id = model_name.split("/", 1)[1]
        return ChatOpenAI(model=model_id, **kwargs)
    elif model_name.startswith("google/"):
        model_id = model_name.split("/", 1)[1]
        return ChatGoogleGenerativeAI(model=model_id, **kwargs)
    elif model_name.startswith("mistral/"):
        model_id = model_name.split("/", 1)[1]
        return ChatMistralAI(model=model_id, **kwargs)
    else:
        raise ValueError(f"Unknown model provider in '{model_name}'")




In [10]:
chain = load_chat_model("google")
chain.invoke( "What is the capital of France?")

AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-001', 'safety_ratings': []}, id='run--46b70e6a-b204-4c0d-ae66-96a560282055-0', usage_metadata={'input_tokens': 7, 'output_tokens': 9, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}})

In [11]:
from typing import Any
from langchain.tools import BaseTool


class AddTool(BaseTool):
    name: str = "add"
    description: str = "Adds two numbers together."

    def _run(self, a: float, b: float) -> float:
        return a + b

    async def _arun(self, a: float, b: float) -> float:
        return self._run(a, b)
# Example usage:
# model = load_chat_model("openai/gpt-4.1-mini")
def load_tool(tool_name: str = None, **kwargs) -> Any:
    """
    Load a tool by name.
    Args:
        tool_name (str, optional): Name of the tool, e.g., "add".
    Returns:
        Any: An instance of the loaded tool.
    """
    if tool_name is None:
        raise ValueError(
            "No tool_name specified. Please provide a tool_name, e.g., 'add'."
        )

    tool_name = tool_name.lower()
    if tool_name == "add":
        return AddTool(**kwargs)
    else:
        raise ValueError(f"Unknown tool '{tool_name}'.")


def get_tools(tool_names: list[str]):
    """Returns a list of tools based on the tool names."""
    tools = [load_tool(tool_name) for tool_name in tool_names]
    return tools

In [16]:
from typing import List, Literal, Annotated
from pydantic import BaseModel, Field


class Configuration(BaseModel):
    """The configuration for the agent"""

    system_prompt: str = Field(
        default="You are a helpful AI assistant",
        description="The system prompt to use for the agent's interactions"
        "This prompt sets the context and behavior of the agent.",
    )
    model: Annotated[
        Literal["openai/gpt-3.5-turbo"],
        {"__template_metadata__": {"kind": "llm"}},
        Field(
            default="openai/gpt-3.5-turbo",
            description="The name of the language model to use for the agents main interactions"
            "should be in the format 'provider/model-id', e.g., 'openai/gpt-3.5-turbo', 'google/gemini-pro', 'mistral/mistral-7b'.",
        ),
    ]
    selected_tools: List[
        Literal["search", "calculator", "code_interpreter", "web_scraper", "add"]
    ] = Field(
        default=["calculator"],
        description="The list of tools to use for the agent's interactions"
        "this list should contain the names of the tools that the agent can use to assist with tasks.",
    )


In [17]:

from langgraph.prebuilt import create_react_agent


from langchain_core.runnables import RunnableConfig


_set_environment_variables(langchain_project="langgraph-assistant")

def make_graph(config: RunnableConfig):
    configurable = config.get("configurable", {})

    llm = configurable.get("model", "google/gemini-2.0-flash-001")
    selected_tools = configurable.get("selected_tools", ["add"])
    prompt = configurable.get("system_prompt", "You are a helpful AI assistant.")

    name = configurable.get("name", "ready_agent")

    graph = create_react_agent(
        name=name,
        model=load_chat_model(llm),
        tools=get_tools(selected_tools),
        prompt=prompt,
        config_schema=Configuration,
    )

    return graph


In [18]:


def main():
    config: RunnableConfig = {
        "configurable": {
            "model": "google/gemini-2.0-flash-001",
            "selected_tools": ["add"],
            "system_prompt": "You are a helpful AI assistant.",
            "name": "MyAgent"
        }
    }

    graph =  make_graph(config)

    # Example usage:  Send a message to the agent
    response = graph.invoke({"input": "What is 2 + 2?"})  # Use ainvoke for async execution
    print(response)

In [19]:
main()

ChatGoogleGenerativeAIError: Invalid argument provided to Gemini: 400 * GenerateContentRequest.contents: contents is not specified
